# Summarize Batch By Parameter Set

Create one summary figure per `parameter_set_*` folder by averaging across repeated simulations with different random seeds.

In [ ]:
from pathlib import Path
import pandas as pd

from parameter_set_batch_summary import summarize_parameter_set_batch, summarize_rate_isi_batch, export_parameter_set_summary_table

In [ ]:
# CONFIGURATION 
BATCH_ROOT_DIR = Path("$$$_sim_batches_posterior_predictive_checks_TA_NOGO") # Path("sim_batches_TEST_v2")
OUTPUT_DIR = "$$$_sim_batches_posterior_predictive_checks_TA_NOGO/parameter_set_summaries_FULL_subset" # None  # None -> create/use BATCH_ROOT_DIR / "parameter_set_summaries"

# Optional subset selection.
# Leave both as None to analyze every parameter_set_* folder.
# Inclusive range example: (0, 10) -> parameter_set_0000 ... parameter_set_0010
SELECTED_PARAMETER_SET_IDS = None
PARAMETER_SET_ID_RANGE = None # (0,9) # None # (0,1) # (0,1) = only the first two sims (for testing purpose) # None
SHOW_PROGRESS = True
XLIM_REL_CUE_S = (-4.0, 1.0)

# Plot options
SHOW_INDIVIDUAL_TRACES = False
SHOW_STD_BAND = True
SHOW_REFERENCE_STD_LINES = True
FIX_YLIM_ACROSS_PARAMETER_SETS = True

# Diagnostic figure export. Turn these off if you only want the CSV/PKL outputs.
EXPORT_TRACE_SUMMARY_FIGURES = True # False # True
EXPORT_RATE_ISI_SUMMARY_FIGURES = False # True

# FEATURE EXTRACTION ON THE MEAN BATCH TRACES
FEATURE_EXTRACTION_CONFIG = {
    'enabled': True,  # Master switch for fitted-baseline residual feature extraction on the batch-mean CST LF, alpha modulation, beta modulation, and synchrony traces.
    'fitting_window': (-3.0, 1.0), # (-4.0, 1.0),  # Cue-relative interval over which the baseline spline is fit and evaluated.
    'analysis_window': (0.0, 0.5),  # Cue-relative interval in which the signed mean residual feature is measured after subtracting the fitted baseline.
    'buffer_s': 0.05,  # Samples inside analysis_window ± buffer_s are excluded from the spline fit, but the feature itself is still computed on analysis_window only.
    'spline_smoothing': None,  # Smoothing strength for UnivariateSpline. None uses the project default heuristic; larger values force a smoother, less flexible baseline.
    'spline_order': 4, # 3  # Spline degree. 3 means cubic spline, which is the default baseline model used in the feature-extraction overlay.
    'n_knots': None,  # If set, fit an LSQ spline with this many interior knots instead of the smoothing-spline path. Leave as None unless you explicitly want knot-controlled flexibility.
    'trend_pre_window': (-3.0, -1.0), # (-4.0, 0.0),  # Cue-relative pre-cue interval on the fitted spline used for the slow-trend feature: mean(fitted post window) - mean(fitted pre window).
    'trend_post_window': (0.0, 1.0),  # Cue-relative post-cue interval on the fitted spline used for the slow-trend feature. This affects OBS_TREND_FEATURE exports and the split fitted-line overlay.
    'extrapolate': False,  # If False, the spline is not allowed to extrapolate outside its support. This mainly matters if you later choose fitting/display windows near the data edges.
}

# Final per-parameter-set table export.
# Missing analyses are exported as NaNs.
EXPORT_SUMMARY_TABLE = True
EXPORT_SUMMARY_CSV_NAME = "parameter_set_feature_summary.csv"
EXPORT_SUMMARY_PKL_NAME = "parameter_set_feature_summary.pkl"

# Optional color overrides for the summary figures.
# Leave empty to reuse the colors saved with each simulation.
COLOR_OVERRIDES = {
    # "lf": "#2ca02c",
    # "alpha": "#1f77b4",
    # "beta": "#9467bd",
    # "common": "#111111",
    # "cst": "#ff7f0e",
    "sync": "#852680",
    # "envelope": "#444444",
    # "burst": "#FF0080",
    # "cue": "#7a7a7a",
}

# Visual style parameters.
PLOT_STYLE = {
    "figure_size": (19, 11),
    "figure_dpi": 120,
    "individual_trace_lw": 0.8,
    "individual_trace_alpha": 0.10,
    "mean_trace_lw": 1.6,
    "std_band_alpha": 0.18,
    "envelope_lw": 1.0,
    "envelope_alpha": 0.0,
    "burst_kernel_lw": 1.1,
    "burst_kernel_alpha": 0.95,
    "reference_std_lw": 1.2,
    "reference_std_alpha": 1.0,
    "burst_epoch_alpha": 0.10,
    "burst_center_lw": 1.1,
    "burst_center_alpha": 0.95,
    "get_ready_lw": 1.0,
    "get_ready_alpha": 0.8,
    "go_cue_lw": 1.1,
    "go_cue_alpha": 0.8,
    "feature_baseline_lw": 1.2,
    "feature_baseline_ls": "--",
    "feature_baseline_alpha": 1.0,
    "feature_baseline_color": "#111111",
    "feature_outside_alpha": 1.0,
    "feature_outside_color": "#DBDBDB",
    "feature_analysis_mean_lw": 1.2,
    "feature_analysis_mean_ls": "--",
    "feature_analysis_mean_alpha": 1.0,
    "feature_text_fontsize": 10,
    "feature_text_weight": "bold",
    "feature_text_box_alpha": 0.75,
    "feature_text_box_facecolor": "white",
}

# Visual parameters for the firing-rate / ISI-CV batch-moment figure.
DISTRIBUTION_COLORS = {
    "firing_rate": "#E67E22",
    "isi_cv": "#00DA5B",
}
DISTRIBUTION_STYLE = {
    "figure_size": (14, 7),
    "figure_dpi": 130,
    "scatter_alpha": 0.55,
    "scatter_size": 22,
    "hist_alpha": 0.75,
    "hist_edgecolor": "white",
    "title_fontsize": 12,
    "panel_title_fontsize": 9,
}

In [ ]:
parameter_set_summary_df = summarize_parameter_set_batch(
    root_dir=BATCH_ROOT_DIR,
    output_dir=OUTPUT_DIR,
    show_individual_traces=SHOW_INDIVIDUAL_TRACES,
    show_std_band=SHOW_STD_BAND,
    show_reference_std_lines=SHOW_REFERENCE_STD_LINES,
    fixed_ylim_across_parameter_sets=FIX_YLIM_ACROSS_PARAMETER_SETS,
    color_overrides=COLOR_OVERRIDES,
    plot_style=PLOT_STYLE,
    selected_parameter_set_ids=SELECTED_PARAMETER_SET_IDS,
    parameter_set_id_range=PARAMETER_SET_ID_RANGE,
    show_progress=SHOW_PROGRESS,
    xlim_rel_cue_s=XLIM_REL_CUE_S,
    feature_config=FEATURE_EXTRACTION_CONFIG,
    export_figures=EXPORT_TRACE_SUMMARY_FIGURES,
)
parameter_set_summary_df

In [ ]:
rate_isi_metrics_df, rate_isi_figures_df = summarize_rate_isi_batch(
    root_dir=BATCH_ROOT_DIR,
    output_dir=OUTPUT_DIR,
    distribution_style=DISTRIBUTION_STYLE,
    distribution_colors=DISTRIBUTION_COLORS,
    selected_parameter_set_ids=SELECTED_PARAMETER_SET_IDS,
    parameter_set_id_range=PARAMETER_SET_ID_RANGE,
    show_progress=SHOW_PROGRESS,
    export_figures=EXPORT_RATE_ISI_SUMMARY_FIGURES,
)
rate_isi_figures_df
rate_isi_metrics_df.head()

In [ ]:
if EXPORT_SUMMARY_TABLE:
    export_summary_df, export_paths = export_parameter_set_summary_table(
        root_dir=BATCH_ROOT_DIR,
        output_dir=OUTPUT_DIR,
        parameter_set_summary_df=globals().get("parameter_set_summary_df"),
        rate_isi_metrics_df=globals().get("rate_isi_metrics_df"),
        feature_config=FEATURE_EXTRACTION_CONFIG,
        selected_parameter_set_ids=SELECTED_PARAMETER_SET_IDS,
        parameter_set_id_range=PARAMETER_SET_ID_RANGE,
        csv_name=EXPORT_SUMMARY_CSV_NAME,
        pickle_name=EXPORT_SUMMARY_PKL_NAME,
    )
    print(export_paths)
    export_summary_df
else:
    print("EXPORT_SUMMARY_TABLE is False; skipping CSV/PKL export.")